# ModernBERT-JEPA decider v2 (masked answer)

**Run 8, one pass through:** the two diagnostics, then two arms of C1 from the same pretrained weights, each with one learning-rate schedule over 2 passes: **A**, tasksource only; **B**, tasksource and our questions shuffled together (no separate stage). Each is scored at four points: questions it trains on, our dev and JevBench. JevBench is the fair judge between A and B (our dev comes from our own generators). C2 (JEPA) stays off.

One question per sequence: `[CLS] question [SEP] state [SEP] A: option … [SEP] Answer: [MASK] [SEP]`.

- **Decision:** a fixed 38-way head at `[MASK]` over the labels A–Z, yes / no and 0–9; each question keeps only its own labels (a choice its first K letters, a statement yes / no, a score its first K digits) before the softmax. The head starts from ModernBERT's pretrained output rows for those 38 tokens and is fully trainable. This is the readout our 4B decoder uses, on an encoder.
- **JEPA (as in I-JEPA):** a target encoder, an EMA copy with no gradients, reads the same sequence with the gold label in place of `[MASK]`; a predictor maps the trained encoder's `[MASK]` vector to that target's vector at the same position.
- **Control:** the same model is trained twice from the pretrained weights: **C1** without the JEPA loss (the supervised baseline), **C2** with it.

The JEPA target carries supervised information (the gold label is in the target's input): this is a JEPA-style predictive objective with a structured target, not the unsupervised image recipe. `01_jepa_quick.ipynb` stays unchanged as the record of runs 1 and 2.

Data: **Stage 1**, a filtered sample of public typed decisions (`tasksource/tasksource-jev-typed-decisions`: commercial-use licences, nothing overlapping either benchmark); **Stage 2**, a subset of our generated data. JevBench's 231 public items are scored untrained and after each stage, never trained on.

## Parameters

In [ ]:
import os

def env(name, default):
    return os.environ.get(name, default)

ENCODER = env("ENCODER", "answerdotai/ModernBERT-base")
DATASET = "skamalj/signpost-d-decisions"           # our generated data (private): train and dev
JEVBENCH_COMMIT = "609a664e81f46e03731600bd7bd9f1b7a61257c5"   # pinned public items
TASKSOURCE = "tasksource/tasksource-jev-typed-decisions"       # Stage 1: public typed decisions
TS_QUESTIONS = int(env("TS_QUESTIONS", "50000"))   # tasksource sample size
TS_PER_SOURCE = int(env("TS_PER_SOURCE", "400"))   # at most this many from one source
TS_BUFFER = 50000                                  # streaming shuffle buffer
OVERLAP_N = 8                                      # drop a state sharing an 8-word run with a test
TRAIN_QUESTIONS = int(env("TRAIN_QUESTIONS", "0"))  # our questions to train on; 0 = all 12,274
MAX_OPTIONS = 26                                   # letters A-Z
BATCH = int(env("BATCH", "16"))                    # questions per step
TRAIN_MAX_LEN = 1024
EVAL_MAX_LEN = 8192                                # JevBench's hard items are long documents
OPTION_MAX_TOKENS = 64                             # each option's text is cut to this
EVAL_TOKEN_BUDGET = 16384                          # tokens per evaluation batch
LR_ENCODER = float(env("LR_ENCODER", "5e-5"))      # top layer; lower layers decay by LLRD
LLRD = float(env("LLRD", "0.95"))                  # gentle, so the whole encoder trains
ARM_EPOCHS = int(env("ARM_EPOCHS", "2"))           # passes over each arm's training data
CURVE_POINTS = 4                                   # scored at each quarter of an arm's training
SANITY_TRAIN, SANITY_TEST = 300, 200               # the sanity test: the answer is in the state
OVERFIT_N, OVERFIT_EPOCHS, OVERFIT_EVERY = 200, 30, 5   # the fair overfit check
OVERFIT_LR = 5e-5                                  # constant, after a short warm-up
LR_HEAD = float(env("LR_HEAD", "5e-4"))            # predictor, temperatures, biases
WARMUP = 0.05
EMA_START, EMA_END = 0.996, 0.9999                 # target-encoder momentum, cosine schedule
LAMBDA_JEPA, LAMBDA_ORD = 0.5, 0.5
LAMBDA_VAR = float(env("LAMBDA_VAR", "0"))         # variance regulariser on ẑ (ablation)
LAMBDA_COV = float(env("LAMBDA_COV", "0"))         # covariance term (ablation; noisy at batch 16)
SEED = int(env("SEED", "42"))

## Packages and the Hugging Face token

The token reads our private dataset: from the environment, a Kaggle or Colab secret, or a hidden prompt. Only where it came from is printed.

In [2]:
!pip install -q "transformers>=4.48"

import copy, gc, json, math, random, re, statistics, time, urllib.request
from collections import Counter, defaultdict

import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from transformers import AutoModelForMaskedLM, AutoTokenizer


def hf_token():
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"], "the environment"
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN"), "Kaggle secret"
    except Exception:
        pass
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token:
            return token, "Colab secret"
    except Exception:
        pass
    import getpass
    token = getpass.getpass("Hugging Face token (read access): ").strip()
    return (token, "the prompt") if token else (None, "NOT FOUND")


HF_TOKEN, token_from = hf_token()
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
print(f"Hugging Face token: {token_from}")

Hugging Face token: the prompt


## Device and log

In [3]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# bf16 where the GPU has it; fp32 otherwise (ModernBERT overflows in fp16)
BF16 = DEVICE.type == "cuda" and torch.cuda.is_bf16_supported()


def log(message):
    print(f"[{time.strftime('%H:%M:%S')}] {message}", flush=True)


gpu = torch.cuda.get_device_name() if DEVICE.type == "cuda" else "CPU"
log(f"torch {torch.__version__} | transformers {transformers.__version__} | {gpu} | "
    f"{'bf16' if BF16 else 'fp32'} | {ENCODER}")

[10:38:17] torch 2.11.0+cu130 | transformers 5.21.0 | NVIDIA RTX PRO 6000 Blackwell Server Edition | bf16 | answerdotai/ModernBERT-base


## One item per question

Every question becomes: question text, state text, option texts and a gold distribution over the options. `noul` options are the criteria's `true` / `false` descriptions (or nothing), shown as `yes` / `no`; `choice` options are `name - description`, shown as `A`, `B`, …; `score` levels are shown as `0`, `1`, …

In [4]:
def state_text(state):
    return state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)


def option_texts(q):
    """-> (option names, option texts) for one System One question."""
    kind, crit = q["type"], q.get("criteria")
    if kind == "noul":
        crit = crit or {}
        return ["true", "false"], [crit.get("true") or "", crit.get("false") or ""]
    if kind == "choice":
        names = list(crit)
        return names, [f"{n} - {d}" if d else n for n, d in crit.items()]
    return [str(i) for i in range(len(crit))], [str(level) for level in crit]


def make_item(kind, question, state, q, gold, source, family=""):
    names, texts = option_texts(q)
    total = sum(gold)
    return {"type": kind, "question": question or "", "state": state_text(state), "names": names,
            "texts": texts, "gold": [g / total for g in gold], "source": source, "family": family}

## Our training and dev data

In [5]:
def load_split(split):
    path = hf_hub_download(DATASET, f"data/{split}.jsonl", repo_type="dataset", token=HF_TOKEN)
    items = []
    for line in open(path, encoding="utf-8"):
        rec = json.loads(line)
        for qid, q in rec["questions"].items():
            probs = rec["gold"][qid]["probabilities"]
            names, _ = option_texts(q)
            if len(names) > MAX_OPTIONS:
                continue
            gold = [float(probs[n]) for n in names]
            items.append(make_item(q["type"], q.get("instructions"), rec["state"], q, gold, split))
    return items


train_all, dev_items = load_split("train"), load_split("dev")
train_items = (random.Random(SEED).sample(train_all, min(TRAIN_QUESTIONS, len(train_all)))
               if TRAIN_QUESTIONS else train_all)
print(f"train {len(train_items)} of {len(train_all)} questions "
      f"{dict(Counter(it['type'] for it in train_items))} | dev {len(dev_items)}")

train.jsonl:   0%|          | 0.00/8.15M [00:00<?, ?B/s]

dev.jsonl:   0%|          | 0.00/661k [00:00<?, ?B/s]

train 12274 of 12274 questions {'score': 4762, 'noul': 3908, 'choice': 3604} | dev 1000


## JevBench public items (test only, never trained on)

One question per item and one expected label, so the gold is one-hot.

In [6]:
def load_jevbench(split):
    url = (f"https://raw.githubusercontent.com/fstandhartinger/jevbench/{JEVBENCH_COMMIT}"
           f"/datasets/public/{split}.jsonl")
    items = []
    for line in urllib.request.urlopen(url).read().decode("utf-8").splitlines():
        rec = json.loads(line)
        q = rec["question"]
        names, _ = option_texts(q)
        if q["type"] == "noul":
            target = "true" if rec["expected"] == "yes" else "false"
        else:
            target = str(rec["expected"])
        gold = [1.0 if n == target else 0.0 for n in names]
        assert sum(gold) == 1, rec["id"]
        items.append(make_item(q["type"], q["instructions"], rec["state"], q, gold,
                               split, rec["family"]))
    return items


jev_items = [it for split in ("original", "easy", "hard") for it in load_jevbench(split)]
print(f"JevBench public: {len(jev_items)} items {dict(Counter(it['source'] for it in jev_items))} "
      f"{dict(Counter(it['type'] for it in jev_items))}")

JevBench public: 231 items {'original': 72, 'easy': 48, 'hard': 111} {'noul': 74, 'choice': 139, 'score': 18}


## Stage 1 data: public typed decisions, filtered

A guard set of word 8-grams from every test state we care about (JevBench's public items, Typed Decisions' train and test splits). Then the tasksource stream, shuffled: rows whose `license_use` is commercial, with 2–26 options (2–10 levels for `score`), at most `TS_PER_SOURCE` per source, and no 8-gram shared with the guard set.

In [7]:
WORD = re.compile(r"\w+")


def ngrams(text, n=OVERLAP_N):
    words = WORD.findall(text.lower())
    return {" ".join(words[i:i + n]) for i in range(len(words) - n + 1)}


guard = set()
for it in jev_items:
    guard |= ngrams(it["state"])
for split in ("train", "test"):
    path = hf_hub_download("LocalLLaMA/typed-decisions", f"all/{split}-00000-of-00001.parquet",
                           repo_type="dataset")
    for state in pd.read_parquet(path)["state"]:
        guard |= ngrams(state_text(json.loads(state)))
print(f"guard set: {len(guard):,} 8-grams")

all/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  599kB            

all/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

all/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  222kB            

all/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

guard set: 162,767 8-grams


In [8]:
def ts_item(row):
    """A tasksource row -> one item, or None if it does not fit."""
    kind, opts, target = row["kind"], row["options"], row["target"]
    if kind == "noul":
        q, gold = {"type": "noul", "criteria": None}, [target[0], 1 - target[0]]
    elif kind == "choice" and 2 <= len(opts) <= MAX_OPTIONS and len(set(opts)) == len(opts):
        q, gold = {"type": "choice", "criteria": {o: None for o in opts}}, list(target)
    elif kind == "score" and 2 <= len(opts) <= 10:
        q, gold = {"type": "score", "criteria": list(opts)}, list(target)
    else:
        return None
    if sum(gold) <= 0 or len(gold) != len(option_texts(q)[0]):
        return None
    return make_item(kind, row["question"], row["state"], q, gold, "tasksource", row["source"])


ts_items, per_source, dropped = [], Counter(), Counter()
stream = load_dataset(TASKSOURCE, "default", split="train", streaming=True, token=HF_TOKEN)
stream = stream.shuffle(seed=SEED, buffer_size=TS_BUFFER)
for row in stream:
    if row["license_use"] != "commercial":
        dropped["licence"] += 1
        continue
    if per_source[row["source"]] >= TS_PER_SOURCE:
        continue
    item = ts_item(row)
    if item is None:
        dropped["shape"] += 1
        continue
    if ngrams(item["state"]) & guard:
        dropped["overlap"] += 1
        continue
    ts_items.append(item)
    per_source[row["source"]] += 1
    if len(ts_items) >= TS_QUESTIONS:
        break
print(f"Stage 1: {len(ts_items)} questions from {len(per_source)} sources "
      f"{dict(Counter(it['type'] for it in ts_items))} | dropped {dict(dropped)}")

README.md:   0%|          | 0.00/12.8k [00:00<?, ?B/s]

Stage 1: 50000 questions from 424 sources {'choice': 39062, 'noul': 6525, 'score': 4413} | dropped {'licence': 53698, 'shape': 3554, 'overlap': 28}


## Label tokens

The 38 labels are A–Z, yes / no and 0–9. Each must be one token of ModernBERT's vocabulary (checked here, with the leading space it has after `Answer:`): the head starts from those tokens' pretrained output rows, and the gold label's token fills the slot in C2's target input.

In [9]:
tok = AutoTokenizer.from_pretrained(ENCODER)
CLS, SEP, PAD, MASK = tok.cls_token_id, tok.sep_token_id, tok.pad_token_id, tok.mask_token_id
LABELS = {"choice": [chr(65 + i) for i in range(MAX_OPTIONS)],
          "noul": ["yes", "no"],
          "score": [str(i) for i in range(10)]}


def one_token(label):
    ids = tok(" " + label, add_special_tokens=False)["input_ids"]
    assert len(ids) == 1, f"label {label!r} is {len(ids)} tokens"
    return ids[0]


LABEL_IDS = {kind: [one_token(lab) for lab in labs] for kind, labs in LABELS.items()}
HEAD_TOKENS = LABEL_IDS["choice"] + LABEL_IDS["noul"] + LABEL_IDS["score"]   # the 38 head outputs
HEAD_SLOTS = {"choice": list(range(26)), "noul": [26, 27], "score": list(range(28, 38))}
print({k: v[:4] for k, v in LABEL_IDS.items()}, "| head outputs", len(HEAD_TOKENS))

config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

{'choice': [329, 378, 330, 399], 'noul': [4754, 642], 'score': [470, 337, 374, 495]} | head outputs 38


## One sequence per question

`[CLS] question [SEP] state [SEP] A: option … [SEP] Answer: [MASK] [SEP]`. The question and the options are kept whole (each option cut at `OPTION_MAX_TOKENS`); if the sequence is too long, the state's end is cut, and the cut is counted. With `gold_fill`, the gold label's token stands in place of `[MASK]` (the JEPA target's input). In training, choice options are shuffled, so a letter never carries a fixed answer.

In [10]:
def ids(text):
    return tok(text, add_special_tokens=False)["input_ids"]


def shuffled(item, rng):
    """A copy of a choice item with its options in a random order (gold moved with them)."""
    if item["type"] != "choice" or rng is None:
        return item
    order = list(range(len(item["names"])))
    rng.shuffle(order)
    return {**item, "names": [item["names"][i] for i in order],
            "texts": [item["texts"][i] for i in order], "gold": [item["gold"][i] for i in order]}


def sequence(item, max_len):
    """-> (token ids with [MASK], mask position, gold label token, state was cut)."""
    labels = LABELS[item["type"]]
    tail = []
    for label, text in zip(labels, item["texts"]):
        line = f"{label}: {text}".strip() if text else label
        tail += ids(line)[:OPTION_MAX_TOKENS] + [SEP]
    tail += ids("Answer:")
    q = ids(item["question"])[:256]
    s = ids(item["state"])
    room = max_len - len(q) - len(tail) - 5
    cut = len(s) > room
    s = s[:max(room, 0)]
    seq = [CLS] + q + [SEP] + s + [SEP] + tail
    pos = len(seq)
    seq += [MASK, SEP]
    best = max(range(len(item["gold"])), key=item["gold"].__getitem__)
    gold_label = LABEL_IDS[item["type"]][best]
    return seq, pos, gold_label, cut


def batch_tensors(items, max_len, gold_fill=False):
    rows = [sequence(it, max_len) for it in items]
    width = max(len(r[0]) for r in rows)
    input_ids, att, pos = [], [], []
    for seq, p, gold_label, _ in rows:
        if gold_fill:
            seq = seq[:p] + [gold_label] + seq[p + 1:]
        input_ids.append(seq + [PAD] * (width - len(seq)))
        att.append([1] * len(seq) + [0] * (width - len(seq)))
        pos.append(p)
    as_t = lambda x: torch.tensor(x, device=DEVICE)
    return as_t(input_ids), as_t(att), as_t(pos), sum(r[3] for r in rows)

## The model

- **Encoder** (trained): ModernBERT-base; its pretrained MLM transform turns the `[MASK]` vector into the space of token outputs.
- **Label head** (trained): 38 outputs (A–Z, yes / no, 0–9), started from the pretrained output rows of those tokens; each question keeps only its own labels.
- **Predictor** (trained): an MLP with a residual from the `[MASK]` vector to ẑ.
- **Target encoder** (EMA copy of the encoder, no gradients, always in eval mode): reads the gold-filled sequence; its vector at the answer position is the JEPA target. It is used only in training.

In [11]:
TYPES = ["noul", "choice", "score"]


class MaskedDecider(nn.Module):
    def __init__(self):
        super().__init__()
        self.mlm = AutoModelForMaskedLM.from_pretrained(ENCODER, attn_implementation="sdpa")
        hidden = self.mlm.config.hidden_size
        for p in self.mlm.model.embeddings.parameters():
            p.requires_grad = False
        self.predictor = nn.Sequential(nn.Linear(hidden, 2 * hidden), nn.GELU(),
                                       nn.LayerNorm(2 * hidden), nn.Linear(2 * hidden, hidden))
        self.label_head = nn.Linear(hidden, len(HEAD_TOKENS))
        with torch.no_grad():                         # start from the pretrained output rows
            self.label_head.weight.copy_(self.mlm.decoder.weight[HEAD_TOKENS])
            if self.mlm.decoder.bias is not None:
                self.label_head.bias.copy_(self.mlm.decoder.bias[HEAD_TOKENS])
        self.target = copy.deepcopy(self.mlm.model).requires_grad_(False)

    def mask_vectors(self, input_ids, att, pos):
        h = self.mlm.model(input_ids=input_ids, attention_mask=att).last_hidden_state
        return h[torch.arange(len(pos), device=h.device), pos]

    def label_logits(self, vectors, items):
        """The 38-way head at [MASK]; each question keeps its own labels -> [items, options]."""
        all38 = self.label_head(self.mlm.head(vectors)).float()
        width = max(len(it["names"]) for it in items)
        out = torch.full((len(items), width), -1e4, device=vectors.device)
        for i, it in enumerate(items):
            k = len(it["names"])
            out[i, :k] = all38[i, HEAD_SLOTS[it["type"]][:k]]
        return out

    def train(self, mode=True):
        """Train the online parts; the EMA target encoder always stays in eval mode."""
        super().train(mode)
        self.target.eval()
        return self

    @torch.no_grad()
    def target_vectors(self, input_ids, att, pos):
        h = self.target(input_ids=input_ids, attention_mask=att).last_hidden_state
        return h[torch.arange(len(pos), device=h.device), pos]

## Scores and losses for a batch

KL to the gold; the ordinal error for `score`; with JEPA on, 1 − cos between the predicted vector and the target's vector at the answer position. Variance and covariance regularisers are off by default (ablations).

In [12]:
def forward(model, items, max_len, jepa):
    input_ids, att, pos, cut = batch_tensors(items, max_len)
    v = model.mask_vectors(input_ids, att, pos)
    scores = model.label_logits(v, items)
    extra = {}
    if jepa:
        t_ids, t_att, t_pos, _ = batch_tensors(items, max_len, gold_fill=True)
        extra["target"] = model.target_vectors(t_ids, t_att, t_pos).float()
        extra["z"] = (v + model.predictor(v)).float()
    return scores, extra, cut


def gold_matrix(items, width):
    g = torch.zeros(len(items), width, device=DEVICE)
    for i, it in enumerate(items):
        g[i, :len(it["gold"])] = torch.tensor(it["gold"], device=DEVICE)
    return g


def losses(scores, extra, items):
    g = gold_matrix(items, scores.shape[1])
    logp = F.log_softmax(scores, dim=1)
    dec = (g * (g.clamp(min=1e-12).log() - logp)).sum(1).mean()
    parts = {"dec": dec}
    is_score = torch.tensor([it["type"] == "score" for it in items], device=DEVICE)
    if is_score.any():
        levels = torch.arange(scores.shape[1], device=DEVICE, dtype=torch.float32)
        n = torch.tensor([len(it["gold"]) - 1 for it in items], device=DEVICE).clamp(min=1)
        gap = ((logp.exp() - g) * levels).sum(1).abs() / n
        parts["ord"] = gap[is_score].mean()
    if "z" in extra:
        z = extra["z"]
        parts["jepa"] = (1 - F.cosine_similarity(z, extra["target"], dim=-1)).mean()
        zc = z - z.mean(0)
        if LAMBDA_VAR:
            parts["var"] = F.relu(1 - (zc.var(0) + 1e-4).sqrt()).mean()
        if LAMBDA_COV:
            cov = (zc.T @ zc) / max(len(items) - 1, 1)
            parts["cov"] = (cov - torch.diag(torch.diag(cov))).pow(2).sum() / z.shape[1]
    weights = {"dec": 1.0, "ord": LAMBDA_ORD, "jepa": LAMBDA_JEPA, "var": LAMBDA_VAR,
               "cov": LAMBDA_COV}
    total = sum(weights[k] * v for k, v in parts.items())
    return total, {k: v.item() for k, v in parts.items()}

## The measures

JevBench's: accuracy (argmax), multi-class Brier, top-label ECE over 10 equal-width bins (empty bins left out), ordinal MAE for `score`; and KL for our soft dev gold.

In [13]:
def measure(rows):
    n = len(rows)
    bins = defaultdict(list)
    for r in rows:
        bins[min(int(r["conf"] * 10), 9)].append(r)
    ece = 0.0
    for b in bins.values():
        right = sum(x["ok"] for x in b) / len(b)
        sure = sum(x["conf"] for x in b) / len(b)
        ece += len(b) / n * abs(right - sure)
    out = {"n": n, "acc": round(sum(r["ok"] for r in rows) / n, 4),
           "brier": round(statistics.mean(r["brier"] for r in rows), 4),
           "ece": round(ece, 4), "kl": round(statistics.mean(r["kl"] for r in rows), 4)}
    mae = [r["mae"] for r in rows if r["mae"] is not None]
    if mae:
        out["ordinal_mae"] = round(statistics.mean(mae), 3)
    return out


def row_of(it, p):
    g = it["gold"]
    top, want = max(range(len(p)), key=p.__getitem__), max(range(len(g)), key=g.__getitem__)
    mae = abs(sum(i * x for i, x in enumerate(p)) - want) if it["type"] == "score" else None
    return {"ok": top == want, "conf": p[top], "margin": None,
            "brier": sum((a - b) ** 2 for a, b in zip(p, g)),
            "kl": sum(b * math.log(b / max(a, 1e-9)) for a, b in zip(p, g) if b > 0),
            "mae": mae, "type": it["type"], "source": it["source"], "family": it["family"]}

## Evaluation

Batches by token budget (long items alone), options in their given order, the measures overall and by split, type and family, and the time per item.

In [14]:
@torch.no_grad()
def evaluate(model, items, max_len, name):
    model.eval()
    lengths = [len(sequence(it, max_len)[0]) for it in items]
    order = sorted(range(len(items)), key=lengths.__getitem__)
    rows, cut, start, t0 = [None] * len(items), 0, 0, time.perf_counter()
    while start < len(order):
        batch = [order[start]]
        while (start + len(batch) < len(order)
               and lengths[order[start + len(batch)]] * (len(batch) + 1) <= EVAL_TOKEN_BUDGET):
            batch.append(order[start + len(batch)])
        group = [items[i] for i in batch]
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=BF16):
            scores, _, c = forward(model, group, max_len, jepa=False)
        cut += c
        for i, it, p in zip(batch, group, F.softmax(scores, dim=1).tolist()):
            rows[i] = row_of(it, p[:len(it["gold"])])
        start += len(batch)
    ms = (time.perf_counter() - t0) / len(items) * 1000
    result = {"all": measure(rows), "ms_per_item": round(ms, 1), "states_cut": cut}
    for key in ("source", "type", "family"):
        values = sorted({r[key] for r in rows})
        result[key] = {v: measure([r for r in rows if r[key] == v]) for v in values}
    log(f"{name}: {result['all']} | {result['ms_per_item']} ms/item | states cut {cut}")
    for key in ("source", "type"):
        log("   " + " | ".join(f"{k} {v['acc']}" for k, v in result[key].items()))
    return result

## Optimizer, schedule and EMA

Layer-wise learning-rate decay from the top encoder layer; the MLM transform at the encoder rate; the label head and predictor at a higher rate; warm-up then cosine decay across both stages. After every step the target encoder moves towards the encoder.

In [15]:
def make_optimizer(model, total_steps, lr=None, constant=False):
    lr = lr or LR_ENCODER
    layers = model.mlm.model.layers
    groups = [{"params": [p for p in layer.parameters() if p.requires_grad],
               "lr": lr * LLRD ** (len(layers) - 1 - i)} for i, layer in enumerate(layers)]
    in_layers = {id(p) for layer in layers for p in layer.parameters()}
    rest = [p for p in model.mlm.parameters() if p.requires_grad and id(p) not in in_layers]
    groups.append({"params": rest, "lr": lr})
    extra = list(model.predictor.parameters()) + list(model.label_head.parameters())
    groups.append({"params": extra, "lr": LR_HEAD})
    opt = torch.optim.AdamW(groups, weight_decay=0.01)
    warm = max(1, int(WARMUP * total_steps))

    def factor(s):
        if s < warm:
            return (s + 1) / warm
        if constant:
            return 1.0
        return 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total_steps - warm)))

    return opt, torch.optim.lr_scheduler.LambdaLR(opt, factor)


@torch.no_grad()
def ema_update(model, step, total_steps):
    m = EMA_END - (EMA_END - EMA_START) * (math.cos(math.pi * step / total_steps) + 1) / 2
    for pt, ps in zip(model.target.parameters(), model.mlm.model.parameters()):
        pt.mul_(m).add_(ps.detach(), alpha=1 - m)
    return m

## Training

A plan is a list of batches (`epochs` passes, reshuffled each pass); training runs a plan, or a slice of one, with choice options shuffled per item. Every 50 steps: the decision loss averaged over the last 50 batches, the current batch's losses, the top learning rate and, with JEPA on, cos(prediction, target), the spread and norm of both, and the EMA momentum.

In [16]:
def make_plan(n, epochs, seed):
    """Batches of item indices: `epochs` passes over n items, reshuffled each pass."""
    rng, plan = random.Random(seed), []
    for _ in range(epochs):
        order = list(range(n))
        rng.shuffle(order)
        plan += [order[b:b + BATCH] for b in range(0, n, BATCH)]
    return plan


def train_stage(model, opt, sched, state, items, name, jepa, epochs=1, shuffle_options=True,
                quiet=False, plan=None):
    model.train()
    recent = []
    rng = random.Random(SEED + len(items) + state["step"])
    t0 = time.perf_counter()
    batches = plan if plan is not None else make_plan(len(items), epochs, SEED + len(items))
    for chunk in batches:
        batch = [shuffled(items[i], rng if shuffle_options else None) for i in chunk]
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=BF16):
            scores, extra, _ = forward(model, batch, TRAIN_MAX_LEN, jepa)
        loss, parts = losses(scores, extra, batch)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        params = [p for g in opt.param_groups for p in g["params"]]
        norm = torch.nn.utils.clip_grad_norm_(params, 1.0)
        opt.step()
        sched.step()
        state["step"] += 1
        m = ema_update(model, state["step"], state["total"]) if jepa else 0.0
        recent = (recent + [parts["dec"]])[-50:]
        if state["step"] % 50 == 0 and not quiet:
            terms = " ".join(f"{k} {v:.4f}" for k, v in parts.items())
            more = ""
            if jepa:
                z, t = extra["z"], extra["target"]
                cos = F.cosine_similarity(z, t, dim=-1).mean().item()
                more = (f" | cos {cos:.3f}"
                        f" | z std {z.std(0).mean().item():.3f}"
                        f" norm {z.norm(dim=-1).mean().item():.1f}"
                        f" | t std {t.std(0).mean().item():.3f}"
                        f" norm {t.norm(dim=-1).mean().item():.1f} | ema {m:.5f}")
            top_lr = max(g["lr"] for g in opt.param_groups[:-1])
            log(f"{name} step {state['step']}/{state['total']} | decision loss, last 50 "
                f"{statistics.mean(recent):.4f} | batch: {terms} | grad {norm:.2f} | "
                f"lr {top_lr:.1e}{more}")
    if not quiet:
        log(f"{name} done in {(time.perf_counter() - t0) / 60:.1f} min")

## Checks before training

On a small batch from every question type: the target encoder gets no gradients and stays in eval mode after `train()`; its outputs are deterministic; the EMA step moves it towards the encoder; the gold label never appears in the context input; every loss is finite. Then the parameter counts: the inference model (encoder, MLM transform, label head) apart from the training-only predictor and EMA copy.

In [17]:
def checks():
    model = MaskedDecider().to(DEVICE)
    sample = [next(it for it in train_items if it["type"] == k) for k in TYPES]
    sample += [next(it for it in ts_items if it["type"] == k) for k in TYPES] if ts_items else []
    model.train()
    assert not model.target.training, "target encoder left in training mode"
    assert not any(p.requires_grad for p in model.target.parameters())
    with torch.autocast("cuda", dtype=torch.bfloat16, enabled=BF16):
        scores, extra, _ = forward(model, sample, TRAIN_MAX_LEN, jepa=True)
        again = forward(model, sample, TRAIN_MAX_LEN, jepa=True)[1]["target"]
    assert torch.allclose(extra["target"], again), "target outputs are not deterministic"
    loss, parts = losses(scores, extra, sample)
    assert all(math.isfinite(v) for v in parts.values()), parts
    loss.backward()
    assert all(p.grad is None for p in model.target.parameters()), "target received gradients"
    for it in sample:
        seq, pos, gold_label, _ = sequence(it, TRAIN_MAX_LEN)
        assert seq[pos] == MASK and gold_label not in seq[pos:], "gold label in the context"
    pairs = list(zip(model.target.parameters(), model.mlm.model.parameters()))
    p_t, p_o = next((t, o) for t, o in pairs if o.requires_grad)
    with torch.no_grad():
        p_o.add_(0.01)
    before = (p_t - p_o).abs().sum().item()
    ema_update(model, 1, 10)
    after = (p_t - p_o).abs().sum().item()
    assert after < before, "EMA did not move the target towards the encoder"
    count = lambda ps: sum(p.numel() for p in ps) / 1e6
    inference = count(model.mlm.parameters()) + count(model.label_head.parameters())
    log(f"checks passed | parts {parts} | inference model {inference:.1f} M | "
        f"predictor {count(model.predictor.parameters()):.1f} M | EMA copy (training only) "
        f"{count(model.target.parameters()):.1f} M")
    del model
    gc.collect()
    torch.cuda.empty_cache()


checks()

model.safetensors: reconstructing file:   0%|          |  0.00B /  599MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/137 [00:00<?, ?it/s]

[10:39:06] checks passed | parts {'dec': 1.0259606838226318, 'ord': 0.15634658932685852, 'jepa': 0.3500584065914154} | inference model 149.7 M | predictor 2.4 M | EMA copy (training only) 149.0 M


## Diagnostic 1: the answer is in the state

Made-up questions where the state names the answer (a choice option by name, a statement as true or false, a score level by name). Reading it off needs only the wiring to be right: the input, the option labels, the head and the loss. Trained on 300, scored on 200 others; a correctly wired model should get close to 100%.

In [18]:
WORDS = ["amber", "basalt", "cedar", "delta", "ember", "falcon", "garnet", "harbor", "indigo",
         "juniper", "kelp", "lumen", "marble", "nectar", "onyx", "pepper", "quartz", "raven"]
LEVELS = ["none", "low", "medium", "high", "critical"]


def sanity_items(n, seed):
    rng, items = random.Random(seed), []
    for i in range(n):
        kind = TYPES[i % 3]
        if kind == "choice":
            opts = rng.sample(WORDS, rng.randint(3, 6))
            pick = rng.randrange(len(opts))
            q = {"type": "choice", "criteria": {o: f"route to {o}" for o in opts}}
            state = f"Ticket {rng.randint(100, 999)}: the reviewer picked {opts[pick]}."
            gold = [1.0 if j == pick else 0.0 for j in range(len(opts))]
            question = "Which option did the reviewer pick?"
        elif kind == "noul":
            truth = rng.random() < 0.5
            q = {"type": "noul", "criteria": None}
            said = "true" if truth else "false"
            state = f"Check {rng.randint(100, 999)}: the statement is {said}."
            gold, question = ([1.0, 0.0] if truth else [0.0, 1.0]), "Is the statement true?"
        else:
            levels = LEVELS[:rng.randint(3, 5)]
            pick = rng.randrange(len(levels))
            q = {"type": "score", "criteria": levels}
            state = f"Report {rng.randint(100, 999)}: the recorded level is {levels[pick]}."
            gold = [1.0 if j == pick else 0.0 for j in range(len(levels))]
            question = "What level is recorded?"
        items.append(make_item(kind, question, state, q, gold, "sanity"))
    return items

In [19]:
def sanity_check():
    random.seed(SEED)
    torch.manual_seed(SEED)
    model = MaskedDecider().to(DEVICE)
    train, test = sanity_items(SANITY_TRAIN, SEED), sanity_items(SANITY_TEST, SEED + 1)
    total = math.ceil(len(train) / BATCH) * 3
    opt, sched = make_optimizer(model, total, lr=OVERFIT_LR, constant=True)
    train_stage(model, opt, sched, {"step": 0, "total": total}, train, "sanity", False, epochs=3)
    result = evaluate(model, test, TRAIN_MAX_LEN, "sanity, 200 unseen")
    acc = result["all"]["acc"]
    log(f"sanity check: {acc} | {'PASS' if acc >= 0.95 else 'FAIL: the wiring is wrong'}")
    del model, opt
    gc.collect()
    torch.cuda.empty_cache()
    return result


sanity = sanity_check()

Loading weights:   0%|          | 0/137 [00:00<?, ?it/s]

[10:39:09] sanity step 50/57 | decision loss, last 50 0.2254 | batch: dec 0.0000 ord 0.0000 | grad 0.00 | lr 5.0e-05
[10:39:09] sanity done in 0.0 min
[10:39:09] sanity, 200 unseen: {'n': 200, 'acc': 1.0, 'brier': 0.0, 'ece': 0.0, 'kl': 0.0, 'ordinal_mae': 0.0} | 0.3 ms/item | states cut 0
[10:39:09]    sanity 1.0
[10:39:09]    choice 1.0 | noul 1.0 | score 1.0
[10:39:09] sanity check: 1.0 | PASS


## Diagnostic 2: can it memorise real questions?

200 of our training questions, 30 passes, a constant learning rate after a short warm-up, options in their given order. Accuracy on those same 200 every 5 passes: a model that can learn this readout climbs towards 100%.

In [20]:
def overfit_check():
    random.seed(SEED)
    torch.manual_seed(SEED)
    model = MaskedDecider().to(DEVICE)
    items = train_items[:OVERFIT_N]
    total = math.ceil(len(items) / BATCH) * OVERFIT_EPOCHS
    opt, sched = make_optimizer(model, total, lr=OVERFIT_LR, constant=True)
    state, curve = {"step": 0, "total": total}, []
    for done in range(OVERFIT_EVERY, OVERFIT_EPOCHS + 1, OVERFIT_EVERY):
        train_stage(model, opt, sched, state, items, "overfit", False, epochs=OVERFIT_EVERY,
                    shuffle_options=False, quiet=True)
        acc = evaluate(model, items, TRAIN_MAX_LEN, f"overfit after {done} passes")["all"]["acc"]
        curve.append((done, acc))
    verdict = "PASS" if curve[-1][1] >= 0.9 else "FAIL: it cannot memorise this readout"
    log(f"overfit curve {curve} | {verdict}")
    del model, opt
    gc.collect()
    torch.cuda.empty_cache()
    return curve


overfit_curve = overfit_check()

Loading weights:   0%|          | 0/137 [00:00<?, ?it/s]

[10:39:16] overfit after 5 passes: {'n': 200, 'acc': 0.625, 'brier': 0.1512, 'ece': 0.0766, 'kl': 0.2598, 'ordinal_mae': 0.892} | 1.4 ms/item | states cut 0
[10:39:16]    train 0.625
[10:39:16]    choice 0.6341 | noul 0.7215 | score 0.525
[10:39:21] overfit after 10 passes: {'n': 200, 'acc': 0.86, 'brier': 0.0345, 'ece': 0.198, 'kl': 0.0668, 'ordinal_mae': 0.439} | 1.4 ms/item | states cut 0
[10:39:21]    train 0.86
[10:39:21]    choice 0.8293 | noul 0.8861 | score 0.85
[10:39:27] overfit after 15 passes: {'n': 200, 'acc': 0.925, 'brier': 0.0148, 'ece': 0.2319, 'kl': 0.0337, 'ordinal_mae': 0.385} | 1.4 ms/item | states cut 0
[10:39:27]    train 0.925
[10:39:27]    choice 0.9512 | noul 0.962 | score 0.875
[10:39:32] overfit after 20 passes: {'n': 200, 'acc': 0.955, 'brier': 0.0082, 'ece': 0.2552, 'kl': 0.0194, 'ordinal_mae': 0.398} | 1.4 ms/item | states cut 0
[10:39:32]    train 0.955
[10:39:32]    choice 0.9756 | noul 0.9747 | score 0.925
[10:39:37] overfit after 25 passes: {'n': 200,

## One arm, with a learning curve

From the pretrained weights: one learning-rate schedule over `ARM_EPOCHS` passes of the arm's items; at each quarter, 1,000 of its own training questions, our dev and JevBench.

In [21]:
def point(model, name, label, seen):
    """(label, training questions, dev, JevBench) at one point of the curve."""
    return (label,
            evaluate(model, seen, TRAIN_MAX_LEN, f"{name} train, {label}"),
            evaluate(model, dev_items, TRAIN_MAX_LEN, f"{name} dev, {label}"),
            evaluate(model, jev_items, EVAL_MAX_LEN, f"{name} JevBench, {label}"))


def run(name, items, jepa=False):
    random.seed(SEED)
    torch.manual_seed(SEED)
    model = MaskedDecider().to(DEVICE)
    seen = random.Random(SEED).sample(items, 1000)
    plan = make_plan(len(items), ARM_EPOCHS, SEED + len(items))
    opt, sched = make_optimizer(model, len(plan))
    state = {"step": 0, "total": len(plan)}
    curve = [point(model, name, "untrained", seen)]
    cuts = [round(len(plan) * k / CURVE_POINTS) for k in range(CURVE_POINTS + 1)]
    for k in range(CURVE_POINTS):
        train_stage(model, opt, sched, state, items, name, jepa, plan=plan[cuts[k]:cuts[k + 1]])
        curve.append(point(model, name, f"{(k + 1) * ARM_EPOCHS / CURVE_POINTS:g} passes", seen))
    return model, curve

## Arm A: tasksource only

In [22]:
model_a, curve_a = run("A", ts_items)
del model_a
gc.collect()
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/137 [00:00<?, ?it/s]

[10:39:46] A train, untrained: {'n': 1000, 'acc': 0.316, 'brier': 0.7177, 'ece': 0.173, 'kl': 1.4217, 'ordinal_mae': 1.341} | 1.1 ms/item | states cut 49
[10:39:46]    tasksource 0.316
[10:39:46]    choice 0.2992 | noul 0.457 | score 0.2078
[10:39:48] A dev, untrained: {'n': 1000, 'acc': 0.219, 'brier': 0.2968, 'ece': 0.2464, 'kl': 0.5431, 'ordinal_mae': 0.847} | 1.6 ms/item | states cut 0
[10:39:48]    dev 0.219
[10:39:48]    choice 0.0867 | noul 0.46 | score 0.1375
[10:39:48] A JevBench, untrained: {'n': 231, 'acc': 0.342, 'brier': 0.7601, 'ece': 0.1724, 'kl': 1.4194, 'ordinal_mae': 0.912} | 3.0 ms/item | states cut 0
[10:39:48]    easy 0.2917 | hard 0.4054 | original 0.2778
[10:39:48]    choice 0.2302 | noul 0.5811 | score 0.2222


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (10638 > 8192). Running this sequence through the model will result in indexing errors


[10:39:54] A step 50/6250 | decision loss, last 50 1.4359 | batch: dec 1.6059 ord 0.2710 | grad 15.50 | lr 8.2e-06
[10:40:00] A step 100/6250 | decision loss, last 50 1.2629 | batch: dec 1.4721 ord 0.5980 | grad 11.21 | lr 1.6e-05
[10:40:06] A step 150/6250 | decision loss, last 50 1.3452 | batch: dec 0.9860 ord 0.4322 | grad 5.67 | lr 2.4e-05
[10:40:13] A step 200/6250 | decision loss, last 50 1.3249 | batch: dec 1.3450 ord 0.0353 | grad 4.76 | lr 3.2e-05
[10:40:19] A step 250/6250 | decision loss, last 50 1.3133 | batch: dec 1.4075 | grad 3.40 | lr 4.0e-05
[10:40:25] A step 300/6250 | decision loss, last 50 1.2618 | batch: dec 1.4026 ord 0.4318 | grad 4.05 | lr 4.8e-05
[10:40:31] A step 350/6250 | decision loss, last 50 1.3043 | batch: dec 1.4207 ord 0.3299 | grad 3.72 | lr 5.0e-05
[10:40:36] A step 400/6250 | decision loss, last 50 1.2859 | batch: dec 1.2203 ord 0.4367 | grad 3.03 | lr 5.0e-05
[10:40:42] A step 450/6250 | decision loss, last 50 1.3074 | batch: dec 1.6382 ord 0.4480 

## Arm B: tasksource and our questions, shuffled together

In [23]:
model_b, curve_b = run("B", ts_items + train_items)

Loading weights:   0%|          | 0/137 [00:00<?, ?it/s]

[10:52:39] B train, untrained: {'n': 1000, 'acc': 0.292, 'brier': 0.6465, 'ece': 0.1901, 'kl': 1.2617, 'ordinal_mae': 1.087} | 1.2 ms/item | states cut 37
[10:52:39]    tasksource 0.3078 | train 0.2304
[10:52:39]    choice 0.2768 | noul 0.4489 | score 0.1776
[10:52:41] B dev, untrained: {'n': 1000, 'acc': 0.219, 'brier': 0.2968, 'ece': 0.2464, 'kl': 0.5431, 'ordinal_mae': 0.847} | 1.6 ms/item | states cut 0
[10:52:41]    dev 0.219
[10:52:41]    choice 0.0867 | noul 0.46 | score 0.1375
[10:52:42] B JevBench, untrained: {'n': 231, 'acc': 0.342, 'brier': 0.7601, 'ece': 0.1724, 'kl': 1.4194, 'ordinal_mae': 0.912} | 3.0 ms/item | states cut 0
[10:52:42]    easy 0.2917 | hard 0.4054 | original 0.2778
[10:52:42]    choice 0.2302 | noul 0.5811 | score 0.2222
[10:52:48] B step 50/7786 | decision loss, last 50 1.2516 | batch: dec 1.2854 ord 0.3681 | grad 15.59 | lr 6.6e-06
[10:52:54] B step 100/7786 | decision loss, last 50 1.1205 | batch: dec 1.0250 ord 0.2717 | grad 10.74 | lr 1.3e-05
[10:53:0

## The two curves

Accuracy at each point for the questions each arm trains on, our dev and JevBench (with its splits), next to chance; then the two diagnostics.

In [24]:
def chance(items):
    return round(statistics.mean(1 / len(it["names"]) for it in items), 3)


for label, curve in [("A", curve_a), ("B", curve_b)]:
    print(f"{label:4s}{'point':>12s} {'train':>7s} {'dev':>7s} {'JevBench':>9s} "
          f"{'original':>9s} {'easy':>7s} {'hard':>7s}")
    for where, tr, dv, jv in curve:
        splits = " ".join(f"{jv['source'][s]['acc']:>7.3f}" for s in ("original", "easy", "hard"))
        print(f"{'':4s}{where:>12s} {tr['all']['acc']:>7.3f} {dv['all']['acc']:>7.3f} "
              f"{jv['all']['acc']:>9.3f}  {splits}")
print(f"chance: dev {chance(dev_items)} | JevBench {chance(jev_items)}")
print(f"\nsanity (answer in the state, 200 unseen): {sanity['all']['acc']}")
print(f"overfit curve (passes, accuracy on the 200): {overfit_curve}")
for label, curve in [("A", curve_a), ("B", curve_b)]:
    last = curve[-1][3]
    print(f"{label} JevBench at the end: Brier {last['all']['brier']} | ECE {last['all']['ece']} | "
          f"ordinal MAE {last['all'].get('ordinal_mae')}")
    print(f"{label} by family:", {k: v["acc"] for k, v in last["family"].items()})

A          point   train     dev  JevBench  original    easy    hard
       untrained   0.316   0.219     0.342    0.278   0.292   0.405
      0.5 passes   0.360   0.353     0.325    0.278   0.271   0.378
        1 passes   0.357   0.251     0.316    0.306   0.312   0.324
      1.5 passes   0.409   0.359     0.346    0.278   0.354   0.387
        2 passes   0.417   0.306     0.320    0.264   0.333   0.351
B          point   train     dev  JevBench  original    easy    hard
       untrained   0.292   0.219     0.342    0.278   0.292   0.405
      0.5 passes   0.352   0.289     0.320    0.333   0.250   0.342
        1 passes   0.390   0.456     0.338    0.306   0.292   0.378
      1.5 passes   0.473   0.477     0.372    0.347   0.542   0.315
        2 passes   0.503   0.457     0.394    0.347   0.562   0.351
chance: dev 0.318 | JevBench 0.318

sanity (answer in the state, 200 unseen): 1.0
overfit curve (passes, accuracy on the 200): [(5, 0.625), (10, 0.86), (15, 0.925), (20, 0.955), (25,

## Arm B on Typed Decisions (benchmark train split, never trained on)

The trained arm-B model, still in memory, on the benchmark's `train` split: one sequence per question, each with its case's state, options and gold distribution. No `test` split. For reference on the same split: our 4B decoder (α 0.75 soup) 0.720 accuracy, KL 0.166.

In [25]:
td_path = hf_hub_download("LocalLLaMA/typed-decisions", "all/train-00000-of-00001.parquet",
                          repo_type="dataset")
td_items, skipped = [], 0
for row in pd.read_parquet(td_path).to_dict("records"):
    state, qs, gold = (json.loads(row[k]) for k in ("state", "questions", "gold"))
    for qid, q in qs.items():
        names, _ = option_texts(q)
        probs = gold[qid]["probabilities"]
        if len(names) > MAX_OPTIONS or any(n not in probs for n in names):
            skipped += 1
            continue
        td_items.append(make_item(q["type"], q.get("instructions"), state, q,
                                  [float(probs[n]) for n in names], row["workflow"],
                                  f"{row['workflow']}/{qid}"))
print(f"Typed Decisions train: {len(td_items)} questions "
      f"{dict(Counter(it['type'] for it in td_items))} | skipped {skipped}")
td = evaluate(model_b, td_items, TRAIN_MAX_LEN, "B on Typed Decisions train")
print("by workflow:", {k: v["acc"] for k, v in td["source"].items()})
print(f"chance {statistics.mean(1 / len(it['names']) for it in td_items):.3f} | "
      f"our 4B on this split: 0.720 accuracy, KL 0.166")

Typed Decisions train: 6000 questions {'choice': 1800, 'noul': 1800, 'score': 2400} | skipped 0
[11:24:45] B on Typed Decisions train: {'n': 6000, 'acc': 0.3498, 'brier': 0.2492, 'ece': 0.0406, 'kl': 0.4562, 'ordinal_mae': 0.869} | 1.1 ms/item | states cut 0
[11:24:45]    agent_trace_observability 0.2967 | customer_service 0.3487 | invoice_processing 0.3473 | security_incidents 0.4067
[11:24:45]    choice 0.2644 | noul 0.5339 | score 0.2758
by workflow: {'agent_trace_observability': 0.2967, 'customer_service': 0.3487, 'invoice_processing': 0.3473, 'security_incidents': 0.4067}
chance 0.318 | our 4B on this split: 0.720 accuracy, KL 0.166
